# Voiceprint: enroll the owners, then pick a threshold (Google Colab)

Produces the speaker profiles `friday.voice.voiceprint` consumes, and — the part
that actually matters — a **threshold** chosen from measured score distributions
rather than guessed.

**There is no model to train here.** `ResemblyzerVerifier` wraps a pretrained
encoder; enrollment is an embedding of your samples, not a fit. A notebook that
claimed to train a speaker-ID model would be inventing work. What is genuinely
unknown is where to put the accept/reject line, and that is what this measures.

**Runtime:** CPU is fine. **You need:** 5-10 clips per owner, a few seconds each,
recorded on the device that will do the verifying — a phone mic and a laptop mic
do not sound alike, and a threshold calibrated on one will misjudge the other.


In [ ]:
!pip install -q resemblyzer soundfile numpy
from resemblyzer import VoiceEncoder, preprocess_wav
import numpy as np, glob, os
encoder = VoiceEncoder()
print('ENCODER OK')

## 1. Upload samples

Two folders: `owner_a/` and `owner_b/`. Wav or mp3, a few seconds each, one
speaker per folder, said naturally — reading a passage in a flat voice enrolls a
flat voice.

In [ ]:
from google.colab import files
import os, zipfile
os.makedirs('owner_a', exist_ok=True); os.makedirs('owner_b', exist_ok=True)
print('Upload a zip containing owner_a/ and owner_b/, or use the file browser.')
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('.')
for who in ('owner_a', 'owner_b'):
    print(who, len(glob.glob(f'{who}/*')), 'clips')

## 2. Embed

In [ ]:
def embed_folder(folder):
    paths = sorted(glob.glob(f'{folder}/*'))
    assert paths, f'no clips in {folder}'
    return np.array([encoder.embed_utterance(preprocess_wav(p)) for p in paths]), paths

emb_a, paths_a = embed_folder('owner_a')
emb_b, paths_b = embed_folder('owner_b')
print('owner_a', emb_a.shape, '| owner_b', emb_b.shape)

## 3. Where is the line?

Leave-one-out: each clip is scored against a profile built from the *other*
clips, so no clip helps verify itself. Then every clip is scored against the
other owner. The gap between those two distributions is the only honest basis
for a threshold.

In [ ]:
def profile(embs, skip=None):
    keep = [e for i, e in enumerate(embs) if i != skip]
    p = np.mean(keep, axis=0)
    return p / np.linalg.norm(p)

def score(e, p):
    return float(np.dot(e / np.linalg.norm(e), p))

same, different = [], []
for embs, other in ((emb_a, emb_b), (emb_b, emb_a)):
    for i, e in enumerate(embs):
        same.append(score(e, profile(embs, skip=i)))
    p = profile(embs)
    different.extend(score(e, p) for e in other)

same, different = np.array(same), np.array(different)
print('same speaker    : min %.3f  mean %.3f' % (same.min(), same.mean()))
print('different speaker: max %.3f  mean %.3f' % (different.max(), different.mean()))

gap = same.min() - different.max()
if gap <= 0:
    print('\nOVERLAP: no threshold separates these two voices cleanly.')
    print('Record more, or more varied, samples before trusting verification.')
else:
    suggested = round(float(different.max() + gap / 2), 3)
    print('\nSuggested FRIDAY_VOICEPRINT_THRESHOLD =', suggested)
    print('(midpoint of a %.3f gap; raise it to be stricter about strangers,' % gap)
    print(' lower it if it keeps failing to recognise you)')

## 4. Export the profiles

Saved as raw float32 bytes — the shape `SpeakerVerifier.enroll()` returns, so the
server can load a profile instead of re-enrolling on every boot. Put them
wherever `OwnerIdentity` is pointed.

In [ ]:
for name, embs in (('owner_a', emb_a), ('owner_b', emb_b)):
    p = profile(embs).astype(np.float32)
    open(f'{name}.profile', 'wb').write(p.tobytes())
    print(name, '->', f'{name}.profile', p.nbytes, 'bytes')

from google.colab import files
files.download('owner_a.profile'); files.download('owner_b.profile')